# CascadeFlow vs Router Comparison

Run CascadeFlow over the router test set, apply the same evaluation logic used by the production router, and compare accuracy/cost/latency decisions sample-by-sample.

In [10]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Imports

In [11]:
from __future__ import annotations

import json
import logging
from pathlib import Path

import pandas as pd

from cascadeflow_experiment import (
    ExperimentConfig,
    ModelSpec,
    create_default_config,
    run_experiment_with_logging,
)
from evaluation import Scorer

## Configure dataset + cascade

Point `router_test_path` at the same split your router evaluation uses. Update `MODEL_SPECS` so every locally hosted VLM is represented and costs reflect USD / 1K tokens. The list is sorted by cost to ensure the cascade orders tiers from cheapest to most expensive.

In [12]:
PROJECT_ROOT = Path.cwd().parent.parent
router_test_path = PROJECT_ROOT / "dataset/final_dataset/router_final/router_test_final.parquet"

config = create_default_config()
config.dataset_path = router_test_path
config.cauldron_lookup_path = PROJECT_ROOT / "dataset/which_vlm_data/processed/cauldron_poc_multi.parquet"
config.image_root = PROJECT_ROOT / "dataset/which_vlm_data/images/cauldron"
config.output_dir = PROJECT_ROOT / "dataset/which_vlm_data/results"
config.project_root = PROJECT_ROOT
config.max_samples = None  # keep the full router test split
config.seed = 13
config.generation_max_tokens = 300
config.generation_temperature = 0.1
config.experiment_name = "cascade_vs_router"
config.verbose_agent = False

MODEL_SPECS = [
    ModelSpec(
        name="PatronusAI/glider",
        base_url="http://localhost:8805/v1",
        cost=0.00045,
        provider="vllm",
        temperature=0.05,
        quality_score=0.6,
        speed_ms=600,
        keywords=["vlm", "fast"],
        domains=["diagram_reasoning", "ocr"],
    ),
    ModelSpec(
        name="PatronusAI/glider-pro",
        base_url="http://localhost:8806/v1",
        cost=0.0009,
        provider="vllm",
        temperature=0.08,
        quality_score=0.75,
        speed_ms=900,
        keywords=["vlm", "balanced"],
        domains=["diagram_reasoning", "math"],
    ),
    ModelSpec(
        name="PatronusAI/glider-max",
        base_url="http://localhost:8807/v1",
        cost=0.0018,
        provider="vllm",
        temperature=0.1,
        quality_score=0.85,
        speed_ms=1300,
        keywords=["vlm", "reasoning"],
        domains=["diagram_reasoning", "math", "ocr"],
    ),
    ModelSpec(
        name="PatronusAI/glider-ultra",
        base_url="http://localhost:8808/v1",
        cost=0.0024,
        provider="vllm",
        temperature=0.15,
        quality_score=0.9,
        speed_ms=1600,
        keywords=["vlm", "deep_reasoning"],
        domains=["diagram_reasoning", "math", "science"],
    ),
    ModelSpec(
        name="PatronusAI/glider-research",
        base_url="http://localhost:8809/v1",
        cost=0.003,
        provider="vllm",
        temperature=0.2,
        quality_score=0.95,
        speed_ms=1900,
        keywords=["vlm", "research"],
        domains=["diagram_reasoning", "ocr", "math", "science"],
    ),
]

MODEL_SPECS = sorted(MODEL_SPECS, key=lambda spec: spec.cost)
config.cascade_models = MODEL_SPECS
config

ExperimentConfig(dataset_path=PosixPath('/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/final_dataset/router_final/router_test_final.parquet'), cauldron_lookup_path=PosixPath('/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/which_vlm_data/processed/cauldron_poc_multi.parquet'), image_root=PosixPath('/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/which_vlm_data/images/cauldron'), output_dir=PosixPath('/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/which_vlm_data/results'), cascade_models=[ModelSpec(name='PatronusAI/glider', base_url='http://localhost:8805/v1', cost=0.00045, provider='vllm', temperature=0.05, max_tokens=None, system_prompt=None, quality_score=0.6, speed_ms=600, keywords=['vlm', 'fast'], domains=['diagram_reasoning', 'ocr'], extra={}), ModelSpec(nam

## Inspect router baseline data

In [13]:
router_df = pd.read_parquet(router_test_path)
print(f"Router test samples: {len(router_df):,}")
router_df.head(3)

Router test samples: 13,707


,sample_id,image_path,prompt_raw,router_task,source_dataset,source_config,txt_question_type,txt_has_mc_options,img_width,img_height,...,qwen3_vl_8b_thinking__sample_score,qwen3_vl_8b_thinking__cost,qwen3_vl_8b_thinking__valid_mask,qwen3_vl_8b_thinking__is_correct,qwen3_vl_8b_thinking__score_f1,gemma_3_27b__sample_score,gemma_3_27b__cost,gemma_3_27b__valid_mask,gemma_3_27b__is_correct,gemma_3_27b__score_f1
0,ai2d_00004_bf3d9c5fd30bf304,None,Question: If the Termites in the community bel...,diagram_reasoning,cauldron_ai2d,ai2d,None,True,1100.0,532.0,...,1.150,0.001191,True,True,0.000000,1.150000,0.000060,True,True,0.030303
1,ai2d_00007_e6f58451a22503d1,None,Question: what does the 2nd picture show?\nCho...,diagram_reasoning,cauldron_ai2d,ai2d,None,True,1500.0,1344.0,...,1.150,0.000758,True,True,0.032258,0.093605,0.000042,True,False,0.046512
2,ai2d_00019_e5f934cfc0951972,None,Question: A food web for a ecosystem is shown ...,diagram_reasoning,cauldron_ai2d,ai2d,None,True,305.0,297.0,...,-0.025,0.001104,True,False,0.000000,0.104091,0.000059,True,False,0.072727


## Run CascadeFlow with multiprocessing

In [15]:
# %%time
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')

results_df, samples_df = run_experiment_with_logging(
    config,
    logger=logging.getLogger('cascadeflow.compare'),
    suppress_hf_logs=True,
    num_workers=4,
    chunk_size=32,
)
results_df.head()

2025-11-29 18:45:54,851 | INFO | Preparing samples for CascadeFlow...


['sample_id', 'image_path', 'prompt_raw', 'router_task', 'source_dataset', 'source_config', 'txt_question_type', 'txt_has_mc_options', 'img_width', 'img_height', 'img_aspect_ratio', 'txt_prompt_length_chars', 'txt_prompt_length_words', 'ground_truth', 'ground_truth_type', 'router_best_model_id', 'router_best_model_name', 'router_chosen_perf', 'router_chosen_cost', 'router_soft_p_deepseek_ocr', 'router_soft_p_qwen2_5_vl_3b', 'router_soft_p_qwen2_5_vl_7b', 'router_soft_p_qwen3_vl_8b_thinking', 'router_soft_p_gemma_3_27b', 'deepseek_ocr__sample_score', 'deepseek_ocr__cost', 'deepseek_ocr__valid_mask', 'deepseek_ocr__is_correct', 'deepseek_ocr__score_f1', 'qwen2_5_vl_3b__sample_score', 'qwen2_5_vl_3b__cost', 'qwen2_5_vl_3b__valid_mask', 'qwen2_5_vl_3b__is_correct', 'qwen2_5_vl_3b__score_f1', 'qwen2_5_vl_7b__sample_score', 'qwen2_5_vl_7b__cost', 'qwen2_5_vl_7b__valid_mask', 'qwen2_5_vl_7b__is_correct', 'qwen2_5_vl_7b__score_f1', 'qwen3_vl_8b_thinking__sample_score', 'qwen3_vl_8b_thinking__c

2025-11-29 18:45:55,072 | INFO | Prepared 0 samples (max_samples=None)


""


## Prepare router metrics

Extract per-sample router accuracy/f1 by looking up the column that corresponds to the router's chosen model.

In [ ]:
ROUTER_SUFFIX_MAP = {
    "router_is_correct": "is_correct",
    "router_f1": "score_f1",
    "router_model_cost": "cost",
}

def _router_metric(row, suffix: str):
    model = row.get("router_best_model_name")
    if not isinstance(model, str):
        return None
    col = f"{model}__{suffix}"
    return row.get(col)

router_eval = router_df.copy()
for new_col, suffix in ROUTER_SUFFIX_MAP.items():
    router_eval[new_col] = router_eval.apply(lambda row, s=suffix: _router_metric(row, s), axis=1)

router_eval = router_eval.rename(
    columns={
        "router_best_model_name": "router_model",
        "router_chosen_cost": "router_cost",
    }
)

router_view = router_eval[
    [
        "sample_id",
        "image_path",
        "prompt_raw",
        "router_task",
        "ground_truth",
        "ground_truth_type",
        "router_model",
        "router_is_correct",
        "router_f1",
        "router_cost",
    ]
].copy()
router_view.head(3)

## Post-process CascadeFlow outputs to match router evaluation

In [ ]:
RENAME_MAP = {
    "model_used": "cascade_model",
    "total_cost": "cascade_cost",
    "latency_ms": "cascade_latency",
    "quality_score": "cascade_quality_score",
    "quality_threshold": "cascade_quality_threshold",
}

def _parse_metadata(meta_str: str) -> dict:
    if isinstance(meta_str, str) and meta_str.strip():
        try:
            return json.loads(meta_str)
        except json.JSONDecodeError:
            return {}
    return {}

def _score_prediction(row):
    return Scorer.compute_all_scores(
        pred=row.get("cascade_response_raw") or "",
        gt=row.get("ground_truth") or "",
        gt_type=row.get("ground_truth_type") or "exact",
    )

cascade_df = results_df.rename(columns=RENAME_MAP).copy()
cascade_df["cascade_response_raw"] = cascade_df["raw_response"]
cascade_df["cascade_metadata"] = cascade_df["metadata"].apply(_parse_metadata)
cascade_df["cascade_draft_tokens"] = cascade_df["cascade_metadata"].apply(lambda m: m.get("draft_tokens"))
cascade_df["cascade_verifier_tokens"] = cascade_df["cascade_metadata"].apply(lambda m: m.get("verifier_tokens"))
cascade_df["cascade_total_tokens"] = cascade_df["cascade_metadata"].apply(lambda m: m.get("total_tokens"))

cascade_df = cascade_df.merge(
    router_view[["sample_id", "ground_truth", "ground_truth_type", "router_task"]],
    on="sample_id",
    how="left",
)

score_cols = cascade_df.apply(_score_prediction, axis=1).apply(pd.Series)
score_cols = score_cols.rename(
    columns={
        "is_correct": "cascade_is_correct",
        "score_f1": "cascade_f1",
        "score_mc_letter_match": "cascade_mc_letter_match",
        "pred_answer_letter": "cascade_pred_answer_letter",
        "gt_answer_letter": "cascade_gt_answer_letter",
    }
)
score_cols["cascade_response_normalized"] = score_cols.apply(
    lambda row: Scorer.normalize_text(cascade_df.loc[row.name, "cascade_response_raw"] or ""),
    axis=1,
)

cascade_df = pd.concat([cascade_df, score_cols], axis=1)

cascade_df["cascade_decision_explanation"] = cascade_df.apply(
    lambda row: json.dumps(
        {
            "cascaded": row.get("cascaded"),
            "draft_accepted": row.get("draft_accepted"),
            "routing_strategy": row.get("routing_strategy"),
            "reason": row.get("routing_reason"),
        }
    ),
    axis=1,
)

cascade_view = cascade_df[
    [
        "sample_id",
        "cascade_model",
        "cascade_cost",
        "cascade_latency",
        "cascade_is_correct",
        "cascade_f1",
        "cascade_response_raw",
        "cascade_response_normalized",
        "cascade_draft_tokens",
        "cascade_verifier_tokens",
        "cascade_total_tokens",
        "cascade_decision_explanation",
    ]
].copy()
cascade_view.head(3)

## Merge router vs cascadeflow

In [ ]:
comparison_df = router_view.merge(cascade_view, on="sample_id", how="inner")
comparison_df["match_router_choice"] = comparison_df["router_model"] == comparison_df["cascade_model"]
print(f"Merged samples: {len(comparison_df):,}")
comparison_df.head(3)

## Summary metrics

In [ ]:
summary = {
    "router_accuracy": float(comparison_df["router_is_correct"].mean()),
    "cascade_accuracy": float(comparison_df["cascade_is_correct"].mean()),
    "router_mean_cost": float(comparison_df["router_cost"].mean()),
    "cascade_mean_cost": float(comparison_df["cascade_cost"].mean()),
    "match_rate": float(comparison_df["match_router_choice"].mean()),
}
print(json.dumps(summary, indent=2))

## Accuracy and cost by match/mismatch

In [ ]:
match_breakdown = (
    comparison_df.groupby("match_router_choice")[
        ["sample_id", "router_is_correct", "cascade_is_correct", "router_cost", "cascade_cost"]
    ]
    .agg(
        samples=("sample_id", "count"),
        router_accuracy=("router_is_correct", "mean"),
        cascade_accuracy=("cascade_is_correct", "mean"),
        router_cost_mean=("router_cost", "mean"),
        cascade_cost_mean=("cascade_cost", "mean"),
    )
)
match_breakdown